In [2]:
import pandas as pd
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS

# ==============================================================================
# 1. CONFIGURACIÓN DE CONEXIÓN A INFLUXDB (v2)
# ==============================================================================
INFLUX_URL = "http://192.168.101.241:8086"       # URL/IP donde está desplegado tu InfluxDB
INFLUX_TOKEN = "smartek-token-12345-final"      # Token de API generado en la interfaz de InfluxDB
INFLUX_ORG = "smartek"             # Nombre de la Organización en InfluxDB
INFLUX_BUCKET = "bambu_maintenance"        # Nombre del Bucket de destino

# Archivo CSV generado durante las impresiones
CSV_FILE = "../datos/dataset_bambu_multi_printer.csv"


# ==============================================================================
# 2. FUNCIÓN PRINCIPAL DE MIGRACIÓN
# ==============================================================================
def importar_csv_a_influx():
    """
    Lee el dataset local en CSV y mapea cada fila al protocolo de datos de InfluxDB
    organizando la información en Tags (metadatos de filtrado) y Fields (métricas).
    """
    print(f"📂 Lectura del archivo CSV: '{CSV_FILE}'...")
    try:
        df = pd.read_csv(CSV_FILE)
    except FileNotFoundError:
        print(f"❌ Error: No se encuentra el archivo '{CSV_FILE}'. Verifica la ruta.")
        return

    # Convertir la columna ISO-8601 de texto a objetos datetime nativos
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # Inicializar el cliente oficial de InfluxDB y la API de escritura síncrona
    client = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
    write_api = client.write_api(write_options=SYNCHRONOUS)

    puntos_batch = []
    total_filas = len(df)
    tamanio_lote = 500  # Agrupar lecturas para minimizar la latencia de red

    print(f"🚀 Subiendo {total_filas} registros al bucket '{INFLUX_BUCKET}'...")

    for index, row in df.iterrows():
        # Construcción del registro mediante el Line Protocol de InfluxDB
        punto = (
            Point("bambu_telemetry")  # Measurement: equivalente al nombre de la tabla
            
            # --- TAGS (Cadenas de texto indexadas para agrupar y filtrar rápidamente) ---
            .tag("printer_name", str(row["printer_name"]))    # Tag 1: Nombre de la máquina
            .tag("serial_number", str(row["serial_number"]))  # Tag 2: Número de serie único
            .tag("tray_type", str(row["tray_type"]))          # Material cargado desde el AMS
            .tag("gcode_state", str(row["gcode_state"]))      # Estado de ejecución (RUNNING, etc.)
            
            # --- FIELDS (Métricas numéricas sobre las que hacer gráficas o cálculos) ---
            .field("nozzle_temp", float(row["nozzle_temp"]))
            .field("bed_temp", float(row["bed_temp"]))
            .field("mc_percent", int(row["mc_percent"]))
            .field("layer_num", int(row["layer_num"]))
            .field("total_layer_num", int(row["total_layer_num"]))
            
            # --- TIMESTAMP (Estampa de tiempo precisa en nanosegundos) ---
            .time(row["timestamp"], WritePrecision.NS)
        )
        puntos_batch.append(punto)

        # Enviar el bloque a InfluxDB cada vez que alcance el tamaño límite
        if len(puntos_batch) >= tamanio_lote:
            write_api.write(bucket=INFLUX_BUCKET, org=INFLUX_ORG, record=puntos_batch)
            puntos_batch = []
            print(f"   ⏳ Procesadas {index + 1}/{total_filas} filas...")

    # Enviar los puntos restantes que no hayan completado el último lote
    if puntos_batch:
        write_api.write(bucket=INFLUX_BUCKET, org=INFLUX_ORG, record=puntos_batch)

    print(f"\n✅ Importación finalizada. {total_filas} registros cargados correctamente en InfluxDB.")
    
    # Cerrar la conexión limpia con el servidor
    client.close()


# ==============================================================================
# 3. PUNTO DE ENTRADA
# ==============================================================================
if __name__ == "__main__":
    importar_csv_a_influx()

📂 Lectura del archivo CSV: '../datos/dataset_bambu_multi_printer.csv'...
🚀 Subiendo 15563 registros al bucket 'bambu_maintenance'...
   ⏳ Procesadas 500/15563 filas...
   ⏳ Procesadas 1000/15563 filas...
   ⏳ Procesadas 1500/15563 filas...
   ⏳ Procesadas 2000/15563 filas...
   ⏳ Procesadas 2500/15563 filas...
   ⏳ Procesadas 3000/15563 filas...
   ⏳ Procesadas 3500/15563 filas...
   ⏳ Procesadas 4000/15563 filas...
   ⏳ Procesadas 4500/15563 filas...
   ⏳ Procesadas 5000/15563 filas...
   ⏳ Procesadas 5500/15563 filas...
   ⏳ Procesadas 6000/15563 filas...
   ⏳ Procesadas 6500/15563 filas...
   ⏳ Procesadas 7000/15563 filas...
   ⏳ Procesadas 7500/15563 filas...
   ⏳ Procesadas 8000/15563 filas...
   ⏳ Procesadas 8500/15563 filas...
   ⏳ Procesadas 9000/15563 filas...
   ⏳ Procesadas 9500/15563 filas...
   ⏳ Procesadas 10000/15563 filas...
   ⏳ Procesadas 10500/15563 filas...
   ⏳ Procesadas 11000/15563 filas...
   ⏳ Procesadas 11500/15563 filas...
   ⏳ Procesadas 12000/15563 filas...